# Подготовка датасета

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import json
import random
import re
from collections import Counter

import numpy as np
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

random.seed(42)
np.random.seed(42)

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/processed/plots", exist_ok=True)
os.makedirs("data/golden_set", exist_ok=True)

## Сэмплирование mMARCO

In [ ]:
from datasets import load_dataset

TARGET_SIZE = 60000

ds = load_dataset("unicamp-dl/mmarco", "russian", split="train").shuffle(seed=42)

unique_queries = {}
base_data = []
random_pool = []

for row in tqdm(ds, desc="mmarco scan"):
    q, p, n = row["query"].strip(), row["positive"].strip(), row["negative"].strip()

    # пул источников для random-негативов
    if len(random_pool) < 200000:
        random_pool.append(p)
        random_pool.append(n)

    if q not in unique_queries:
        unique_queries[q] = {"pos": p, "negs": []}

    if len(unique_queries[q]["negs"]) < 3 and n not in unique_queries[q]["negs"]:
        unique_queries[q]["negs"].append(n)
        if len(unique_queries[q]["negs"]) == 3:
            qid = len(base_data)
            base_data.append({
                "query_id": f"q{qid}",
                "query": q,
                "positive_doc_id": f"d{qid}_pos",
                "positive_text": p,
                "hard_negative_ids": [f"d{qid}_hn{i}" for i in range(3)],
                "hard_negatives": list(unique_queries[q]["negs"]),
            })

    if len(base_data) >= TARGET_SIZE:
        break

for item in tqdm(base_data, desc="random negs"):
    forbidden = {item["positive_text"], *item["hard_negatives"]}
    item["random_negatives"] = [r for r in random.sample(random_pool, 10) if r not in forbidden][:2]

with open("data/processed/mmarco_ru_60k.json", "w", encoding="utf-8") as f:
    json.dump(base_data, f, ensure_ascii=False, indent=2)

## Сэмпл и базовая чистка длин

In [ ]:
from transformers import AutoTokenizer

NUM_SAMPLES = 50000

with open("data/processed/mmarco_ru_60k.json", encoding="utf-8") as f:
    raw_data = json.load(f)

ids = list(range(len(raw_data)))
random.shuffle(ids)
queries_data = [raw_data[i] for i in ids[:NUM_SAMPLES]]

base_tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")

def tlen(s):
    return len(base_tokenizer(s, add_special_tokens=False).input_ids)

seen = set()
clean = []
for it in queries_data:
    q = it["query"].strip()
    if not q or q in seen:
        continue
    if not (3 <= tlen(q) <= 64):
        continue
    if tlen(it["positive_text"]) > 480:
        continue
    bad = {it["positive_text"], *it["hard_negatives"]}
    it["random_negatives"] = [r for r in it["random_negatives"] if r not in bad]
    seen.add(q)
    clean.append(it)

queries_data = clean
print(f"После очистки: {len(queries_data)}")

## Подготовка vLLM

In [ ]:
from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen3-32B"

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.9,
    dtype="bfloat16",
    max_model_len=16000,
)
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sampling_params = SamplingParams(temperature=0.7, max_tokens=4096)

SYS_MSG = "Ты ИИ-ассистент, строго следующий инструкциям и возвращающий только JSON."

THINK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)

def parse_json_safe(text):
    if not text:
        return None
    text = THINK_RE.sub("", text).strip()
    text = re.sub(r"```(?:json|JSON)?\s*", "", text)
    text = re.sub(r"\s*```", "", text).strip()
    start = text.find("{")
    if start == -1:
        return None
    depth, in_str, esc = 0, False, False
    end = -1
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
        else:
            if ch == '"': in_str = True
            elif ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
    if end == -1:
        return None
    js = text[start:end].replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    try:
        return json.loads(js)
    except json.JSONDecodeError:
        return None

def build_prompt(user_msg):
    return gen_tokenizer.apply_chat_template(
        [{"role": "system", "content": SYS_MSG},
         {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

## Генерация инструкций

In [ ]:
INSTRUCTION_STYLES = [
    {"tone": "формальный, академический", "length": "2–3 предложения, подробная"},
    {"tone": "лаконичный, телеграфный", "length": "1 короткое предложение"},
    {"tone": "разговорный, неформальный", "length": "1–2 предложения"},
    {"tone": "строгий, бюрократический", "length": "2–3 предложения с перечислением требований"},
    {"tone": "нейтральный, деловой", "length": "1–2 предложения средней длины"},
]

with open("prompts/instruction.txt", encoding="utf-8") as f:
    instruction_prompt = f.read()

prompts = []
styles = []
for item in queries_data:
    style = random.choice(INSTRUCTION_STYLES)
    styles.append(style)
    user_msg = instruction_prompt.format(
        query=item["query"],
        pos_doc=item["positive_text"],
        neg_doc=item["hard_negatives"][0],
        style_hint=f"тон {style['tone']}; длина — {style['length']}",
    )
    prompts.append(build_prompt(user_msg))

outputs = llm.generate(prompts, sampling_params)

for i, out in enumerate(outputs):
    obj = parse_json_safe(out.outputs[0].text)
    instr = obj.get("instruction") if isinstance(obj, dict) else None
    queries_data[i]["generated_instruction"] = instr.strip() if isinstance(instr, str) else None
    queries_data[i]["instruction_style_tone"] = styles[i]["tone"]
    queries_data[i]["instruction_style_length"] = styles[i]["length"]

queries_data = [x for x in queries_data if x.get("generated_instruction")]
print(f"С инструкциями: {len(queries_data)}")

with open("data/processed/train_data_w_inst.json", "w", encoding="utf-8") as f:
    json.dump(queries_data, f, ensure_ascii=False, indent=2)

## Генерация instruction negatives

In [ ]:
with open("prompts/instruction_negative.txt", encoding="utf-8") as f:
    neg_prompt = f.read()

prompts = []
for item in queries_data:
    user_msg = neg_prompt.format(
        query=item["query"],
        instruction=item["generated_instruction"],
        pos_doc=item["positive_text"],
    )
    prompts.append(build_prompt(user_msg))

outputs = llm.generate(prompts, sampling_params)

for i, out in enumerate(outputs):
    obj = parse_json_safe(out.outputs[0].text)
    if isinstance(obj, dict) and obj.get("passage"):
        queries_data[i]["instruction_negative"] = {
            "passage": obj["passage"].strip(),
            "violation_reason": (obj.get("violation_reason") or "").strip(),
        }
    else:
        queries_data[i]["instruction_negative"] = None

queries_data = [x for x in queries_data if x.get("instruction_negative")]
print(f"С instruction negatives: {len(queries_data)}")

with open("data/processed/train_data_w_inst_neg.json", "w", encoding="utf-8") as f:
    json.dump(queries_data, f, ensure_ascii=False, indent=2)

del llm, gen_tokenizer
gc.collect()
torch.cuda.empty_cache()

## Фильтрация по языку

In [ ]:
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 0

with open("data/processed/train_data_w_inst_neg.json", encoding="utf-8") as f:
    queries_data = json.load(f)

def is_ru(text, min_len=50):
    if not text or len(text) < min_len:
        return True
    try:
        return detect(text) == "ru"
    except LangDetectException:
        return False

queries_data = [
    x for x in queries_data
    if is_ru(x["query"], min_len=50)
    and is_ru(x["positive_text"])
    and is_ru(x["generated_instruction"])
    and is_ru(x["instruction_negative"]["passage"])
]
print(f"После langdetect: {len(queries_data)}")

## Cross-encoder скоринг

In [ ]:
from sentence_transformers import CrossEncoder

ce = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=2048, device="cuda")

def with_inst(x):
    return f"{x['generated_instruction']} {x['query']}"

pairs_pos_no   = [[x["query"], x["positive_text"]] for x in queries_data]
pairs_pos_with = [[with_inst(x), x["positive_text"]] for x in queries_data]
pairs_in_no    = [[x["query"], x["instruction_negative"]["passage"]] for x in queries_data]
pairs_in_with  = [[with_inst(x), x["instruction_negative"]["passage"]] for x in queries_data]

s_pos_no   = ce.predict(pairs_pos_no, batch_size=256, show_progress_bar=True)
s_pos_with = ce.predict(pairs_pos_with, batch_size=256, show_progress_bar=True)
s_in_no    = ce.predict(pairs_in_no, batch_size=256, show_progress_bar=True)
s_in_with  = ce.predict(pairs_in_with, batch_size=256, show_progress_bar=True)

hard_pairs = [[x["query"], n] for x in queries_data for n in x["hard_negatives"]]
rand_pairs = [[x["query"], n] for x in queries_data for n in x["random_negatives"]]
s_hard = ce.predict(hard_pairs, batch_size=256, show_progress_bar=True)
s_rand = ce.predict(rand_pairs, batch_size=256, show_progress_bar=True)

hn_idx = rn_idx = 0
for i, x in enumerate(queries_data):
    hn_len = len(x["hard_negatives"])
    rn_len = len(x["random_negatives"])
    x["ce_scores"] = {
        "pos_no": round(float(s_pos_no[i]), 2),
        "pos_with": round(float(s_pos_with[i]), 2),
        "instr_neg_no": round(float(s_in_no[i]), 2),
        "instr_neg_with": round(float(s_in_with[i]), 2),
        "hard_neg": [round(float(s_hard[hn_idx + j]), 2) for j in range(hn_len)],
        "rand_neg": [round(float(s_rand[rn_idx + j]), 2) for j in range(rn_len)],
    }
    hn_idx += hn_len
    rn_idx += rn_len

del ce
gc.collect()
torch.cuda.empty_cache()

## Распределения CE-скоров

In [ ]:
import matplotlib.pyplot as plt

arrs = {
    "pos_no": s_pos_no, "pos_with": s_pos_with,
    "instr_neg_no": s_in_no, "instr_neg_with": s_in_with,
    "hard_neg": np.array(s_hard), "rand_neg": np.array(s_rand),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (name, arr) in zip(axes.flat, arrs.items()):
    ax.hist(arr, bins=40)
    ax.set_title(name)
    ax.axvline(np.median(arr), color="r", linestyle="--", label=f"med={np.median(arr):.2f}")
    ax.legend()
plt.tight_layout()
plt.savefig("data/processed/plots/ce_scores.png", dpi=120)
plt.show()

delta_pos = s_pos_no - s_pos_with
delta_in = s_in_no - s_in_with

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(delta_pos, bins=40); axes[0].set_title("delta pos (no - with)")
axes[1].hist(delta_in,  bins=40); axes[1].set_title("delta instr_neg (no - with)")
plt.tight_layout()
plt.savefig("data/processed/plots/ce_deltas.png", dpi=120)
plt.show()

## Отбор примеров по CE-порогам

In [ ]:
POS_NO_MIN          = 0.25
POS_WITH_MIN        = 0.0
POS_DELTA_MAX       = 0.5
INSTR_NEG_NO_MIN    = 0.3
INSTR_NEG_WITH_MAX  = 0.99
INSTR_NEG_DELTA_MIN = 0.0

final_dataset = []
drop_reasons = {"pos_no": 0, "pos_with": 0, "pos_delta": 0,
                "in_no": 0, "in_with": 0, "in_delta": 0, "dup": 0}

for x in queries_data:
    s = x["ce_scores"]
    pn, pw = s["pos_no"], s["pos_with"]
    in_, iw = s["instr_neg_no"], s["instr_neg_with"]

    if pn < POS_NO_MIN: drop_reasons["pos_no"] += 1; continue
    if pw < POS_WITH_MIN: drop_reasons["pos_with"] += 1; continue
    if (pn - pw) > POS_DELTA_MAX: drop_reasons["pos_delta"] += 1; continue
    if in_ < INSTR_NEG_NO_MIN: drop_reasons["in_no"] += 1; continue
    if iw > INSTR_NEG_WITH_MAX: drop_reasons["in_with"] += 1; continue
    if (in_ - iw) < INSTR_NEG_DELTA_MIN: drop_reasons["in_delta"] += 1; continue

    p = x["positive_text"]
    n = x["instruction_negative"]["passage"]
    if p == n or n in x["hard_negatives"] or n in x["random_negatives"]:
        drop_reasons["dup"] += 1
        continue

    final_dataset.append(x)

print(f"Осталось {len(final_dataset)} из {len(queries_data)}")
print(f"Причины дропа: {drop_reasons}")

## Разделение на train / val / golden

In [ ]:
VAL_SIZE = 500
GOLDEN_SIZE = 100

golden_final, val_final, train_final = [], [], []
seen_positives = set()

for x in reversed(final_dataset):
    pos = x["positive_text"]
    if len(golden_final) < GOLDEN_SIZE:
        if pos not in seen_positives:
            golden_final.append(x)
            seen_positives.add(pos)
    elif len(val_final) < VAL_SIZE:
        if pos not in seen_positives:
            val_final.append(x)
            seen_positives.add(pos)
    else:
        train_final.append(x)

train_final.reverse()
val_final.reverse()
golden_final.reverse()

print(f"train={len(train_final)} val={len(val_final)} golden={len(golden_final)}")

with open("data/processed/train_data_final.json", "w", encoding="utf-8") as f:
    json.dump(train_final, f, ensure_ascii=False, indent=2)
with open("data/processed/val_data_final.json", "w", encoding="utf-8") as f:
    json.dump(val_final, f, ensure_ascii=False, indent=2)

## Проверка утечек

In [ ]:
train_q = {x["query"] for x in train_final}
val_q = {x["query"] for x in val_final}
gold_q = {x["query"] for x in golden_final}
print(f"val→train: {len(val_q & train_q)}")
print(f"gold→train: {len(gold_q & train_q)}")
print(f"gold→val: {len(gold_q & val_q)}")

train_pos = {x["positive_text"] for x in train_final}
print(f"golden positive в train: {sum(1 for x in golden_final if x['positive_text'] in train_pos)}/{len(golden_final)}")

## Выгрузка golden set для фильтрации

In [ ]:
golden_for_review = [{
    "id": i,
    "query": x["query"],
    "instruction": x["generated_instruction"],
    "instruction_style_tone": x["instruction_style_tone"],
    "instruction_style_length": x["instruction_style_length"],
    "positive_text": x["positive_text"],
    "instruction_negative": x["instruction_negative"]["passage"],
    "violation_reason": x["instruction_negative"]["violation_reason"],
    "ce_scores": x["ce_scores"],
    "query_id": x["query_id"],
    "hard_negatives": x["hard_negatives"],
    "random_negatives": x["random_negatives"],
} for i, x in enumerate(golden_final)]

with open("data/golden_set/golden_for_manual_review.json", "w", encoding="utf-8") as f:
    json.dump(golden_for_review, f, ensure_ascii=False, indent=2)

print(f"Выгружено: {len(golden_for_review)} примеров")

## Статистика финального датасета

In [ ]:
stats_fields = {
    "query": lambda x: x["query"],
    "positive": lambda x: x["positive_text"],
    "hard_neg": lambda x: x["hard_negatives"][0],
    "instruction": lambda x: x["generated_instruction"],
    "instr_neg": lambda x: x["instruction_negative"]["passage"],
}

print("Длины")
for name, getter in stats_fields.items():
    v = [tlen(getter(x)) for x in final_dataset]
    print(f"{name:12s} mean={np.mean(v):6.1f}  p50={np.percentile(v, 50):6.0f}  "
          f"p95={np.percentile(v, 95):6.0f}  max={max(v):5d}  >512: {sum(t > 512 for t in v)}")

print("\nУникальность")
print(f"query:       {len(set(x['query'] for x in final_dataset))}")
print(f"positive:    {len(set(x['positive_text'] for x in final_dataset))}")
print(f"instruction: {len(set(x['generated_instruction'] for x in final_dataset))}")

print("\nСтили инструкций")
for k, v in Counter(x["instruction_style_tone"] for x in final_dataset).most_common():
    print(f"  {k}: {v}")

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (name, getter) in zip(axes.flat, stats_fields.items()):
    v = [tlen(getter(x)) for x in final_dataset]
    ax.hist(v, bins=40)
    ax.set_title(f"{name} (tokens)")
    ax.axvline(512, color="r", linestyle="--", label="e5 limit")
    ax.legend()
axes.flat[-1].axis("off")
plt.tight_layout()
plt.savefig("data/processed/plots/lengths.png", dpi=120)
plt.show()

## Сборка golden_clean по вердиктам Gemini + ручной проверки

In [ ]:
with open("data/golden_set/golden_for_manual_review.json", encoding="utf-8") as f:
    golden_raw = json.load(f)
with open("data/golden_set/gemini_verdicts.json", encoding="utf-8") as f:
    verdicts = json.load(f)

verdict_map = {v["id"]: v for v in verdicts}

golden_merged = []
stats = {"total": len(golden_raw), "all_ok": 0, "bad_instruction": 0,
         "pos_not_relevant": 0, "neg_not_violating": 0, "missing_verdict": 0}

for item in golden_raw:
    v = verdict_map.get(item["id"])
    if v is None:
        stats["missing_verdict"] += 1
        continue

    item["verdict_instruction_ok"] = v["instruction_ok"]
    item["verdict_pos_still_relevant"] = v["pos_still_relevant"]
    item["verdict_instr_neg_violates"] = v["instr_neg_violates"]

    if not v["instruction_ok"]:
        stats["bad_instruction"] += 1
    elif not v["pos_still_relevant"]:
        stats["pos_not_relevant"] += 1
    elif not v["instr_neg_violates"]:
        stats["neg_not_violating"] += 1
    else:
        stats["all_ok"] += 1

    golden_merged.append(item)

golden_clean = [x for x in golden_merged if x["verdict_instruction_ok"] 
                and x["verdict_pos_still_relevant"]
                and x["verdict_instr_neg_violates"]]

print(stats)
print(f"Чистый golden: {len(golden_clean)}")

with open("data/golden_set/golden_clean.json", "w", encoding="utf-8") as f:
    json.dump(golden_clean, f, ensure_ascii=False, indent=2)
with open("data/golden_set/golden_merged_all.json", "w", encoding="utf-8") as f:
    json.dump(golden_merged, f, ensure_ascii=False, indent=2)

## Метаданные

In [ ]:
meta = {
    "n_raw": len(raw_data),
    "n_sampled": NUM_SAMPLES,
    "n_after_ce_filter": len(final_dataset),
    "n_train": len(train_final),
    "n_val": len(val_final),
    "n_golden": len(golden_final),
    "thresholds": {
        "POS_NO_MIN": POS_NO_MIN,
        "POS_WITH_MIN": POS_WITH_MIN,
        "POS_DELTA_MAX": POS_DELTA_MAX,
        "INSTR_NEG_NO_MIN": INSTR_NEG_NO_MIN,
        "INSTR_NEG_WITH_MAX": INSTR_NEG_WITH_MAX,
        "INSTR_NEG_DELTA_MIN": INSTR_NEG_DELTA_MIN,
    },
}
with open("data/processed/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)